In [1]:
import pandas as pd
import utils
import os
from Bio import SeqIO
import pickle
import argparse
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from transformers import T5EncoderModel, T5Tokenizer
from config import config
from dataset import MyDataset
from torch import optim, nn
from torch.utils.data import TensorDataset
from torch.optim.lr_scheduler import ReduceLROnPlateau
from sklearn.ensemble import RandomForestClassifier
from model import Cnn
from loss import *
from torch.utils.data import DataLoader
from torch.optim import lr_scheduler
from utils import *
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, matthews_corrcoef
import numpy as np
from sklearn.preprocessing import StandardScaler

In [2]:
def test_imbalance(model, test_dataloader, embedding_df):
    epoch_labels, epoch_preds = [], []
    model.eval()
    for ids, y in test_dataloader:
        x = take_embedding(embedding_df, ids)
        y = y.to(config.device)
        with torch.no_grad():
            outputs = model(x)
        preds = outputs
        epoch_labels += list(y.cpu().numpy())
        epoch_preds += list(preds.argmax(1).cpu().numpy())
    return epoch_labels, epoch_preds

In [3]:
utils.seed_everything(config.seed)
label_dict = {'non-PVP': 0, 'PVP': 1}
thresholds = ["1", "3", "5", "7", "9"]

In [4]:
# get_embedding
embedding_df = pd.read_excel(config.embedding_file)
embedding_df.set_index("id", inplace=True)

In [5]:
file_dir = "./data/imbalance_data"
model_dir = "./model/imbalance_data"
results_dir = "./results/imbalance_data"
for threshold in thresholds:
    test_file = os.path.join(file_dir, "IR_"+threshold, "test.fasta")
    model_file = os.path.join(model_dir, "IR_"+threshold, "model.path")
    results_file = os.path.join(results_dir, "IR_"+threshold, "metric.csv")
    test_proteins, test_ids, test_labels = [], [], []
    
    for rec in SeqIO.parse(test_file, "fasta"):
        id = str(rec.id.split(".")[0].strip())
        seq = str(rec.seq)
        label = str(rec.description.split("#")[-1].strip())
        test_proteins.append(seq)
        test_ids.append(id)
        test_labels.append(label_dict[label])
    
    y_test = np.array(test_labels)
    test_data = MyDataset(test_ids, test_labels)
    test_dataloader = DataLoader(test_data, shuffle=False, batch_size=config.batch_size)
    
    model = Cnn().to(config.device)
    model.load_state_dict(torch.load(model_file))
    # test
    labels, test_epoch_preds = test_imbalance(model, test_dataloader, embedding_df)
    # results
    test_acc = accuracy_score(y_test, test_epoch_preds)
    precision = precision_score(y_test, test_epoch_preds)
    recall = recall_score(y_test, test_epoch_preds)
    f1 = f1_score(y_test, test_epoch_preds)
    mcc = matthews_corrcoef(y_test, test_epoch_preds)
    tn, fp, fn, tp = confusion_matrix(y_test, test_epoch_preds).ravel()
    sensitivity = tp / (tp + fn)
    specificity = tn / (tn + fp)
    print("_________________________result for "+ threshold + "________________________")
    print('test ACC of classifier: %.4f' % (test_acc))
    print('test precision of classifier: %.4f' % (precision))
    print('test recall of classifier: %.4f' % (recall))
    print('test f1 of classifier: %.4f' % (f1))
    print('test mcc of classifier: %.4f' % (mcc))
    print('test sensitivity of classifier: %.4f' % (sensitivity))
    print('test specificity of classifier: %.4f' % (specificity))
    
    df = pd.DataFrame({
        'ACC': [test_acc],
        'Precision': [precision],
        'Recall': [recall],
        'F1': [f1],
        'MCC': [mcc],
        'Sensitivity': [sensitivity],
        'Specificity': [specificity]
    })
    df.to_csv(results_file, index=False)


/home/oyh/anaconda3/envs/pytorch/lib/python3.8/site-packages/torch/nn/modules/lazy.py:178: UserWarning: Lazy modules are a new feature under heavy development so changes to the API or functionality can happen at any moment.
  warnings.warn('Lazy modules are a new feature under heavy development '


_________________________result for 1________________________
test ACC of classifier: 0.9834
test precision of classifier: 0.9816
test recall of classifier: 0.9851
test f1 of classifier: 0.9833
test mcc of classifier: 0.9669
test sensitivity of classifier: 0.9851
test specificity of classifier: 0.9818


/home/oyh/anaconda3/envs/pytorch/lib/python3.8/site-packages/torch/nn/modules/lazy.py:178: UserWarning: Lazy modules are a new feature under heavy development so changes to the API or functionality can happen at any moment.
  warnings.warn('Lazy modules are a new feature under heavy development '


_________________________result for 3________________________
test ACC of classifier: 0.9832
test precision of classifier: 0.9699
test recall of classifier: 0.9624
test f1 of classifier: 0.9661
test mcc of classifier: 0.9550
test sensitivity of classifier: 0.9624
test specificity of classifier: 0.9901


/home/oyh/anaconda3/envs/pytorch/lib/python3.8/site-packages/torch/nn/modules/lazy.py:178: UserWarning: Lazy modules are a new feature under heavy development so changes to the API or functionality can happen at any moment.
  warnings.warn('Lazy modules are a new feature under heavy development '


_________________________result for 5________________________
test ACC of classifier: 0.9867
test precision of classifier: 0.9643
test recall of classifier: 0.9544
test f1 of classifier: 0.9593
test mcc of classifier: 0.9514
test sensitivity of classifier: 0.9544
test specificity of classifier: 0.9931


/home/oyh/anaconda3/envs/pytorch/lib/python3.8/site-packages/torch/nn/modules/lazy.py:178: UserWarning: Lazy modules are a new feature under heavy development so changes to the API or functionality can happen at any moment.
  warnings.warn('Lazy modules are a new feature under heavy development '


_________________________result for 7________________________
test ACC of classifier: 0.9866
test precision of classifier: 0.9566
test recall of classifier: 0.9376
test f1 of classifier: 0.9470
test mcc of classifier: 0.9394
test sensitivity of classifier: 0.9376
test specificity of classifier: 0.9938


/home/oyh/anaconda3/envs/pytorch/lib/python3.8/site-packages/torch/nn/modules/lazy.py:178: UserWarning: Lazy modules are a new feature under heavy development so changes to the API or functionality can happen at any moment.
  warnings.warn('Lazy modules are a new feature under heavy development '


_________________________result for 9________________________
test ACC of classifier: 0.9882
test precision of classifier: 0.9499
test recall of classifier: 0.9328
test f1 of classifier: 0.9412
test mcc of classifier: 0.9347
test sensitivity of classifier: 0.9328
test specificity of classifier: 0.9944
